# Patch Set
In this notebook, we will decided how to generate patchsets (a cache object for patch generation) from a slide.

In [ ]:
from histokit.dataset.dataset import Dataset


cerv_mini_dataset = Dataset.from_index(
    "../data/icaird/cervical_mini/index.csv",
    "../data/icaird/cervical_mini/labels.json",
    slides_dir="slides",
    annotations_dir="annotations"
)

for sample in cerv_mini_dataset.samples():
    print(sample)

In [ ]:
import datetime

import numpy as np
import pandas as pd

from histokit.dataset.sample import Sample
from histokit.io.annotations.annotation import AnnotationSet
from histokit.io.slides.slide import SlideBase
from histokit.patchset.context import PatchContext
from histokit.patchset.manifest import PatchSetManifest
from histokit.patchset.patchset import PatchSet
from histokit.segmentation.detector import Detector
from histokit.segmentation.detectors import per_patch_canny_ranker
from histokit.utils.convert import to_frame_with_locations


def make_patchset_df(
    slide: SlideBase,
    patch_size: int,
    patch_level: int,
    detector: Detector | None = None,
    annotation_set: AnnotationSet | None = None,
) -> pd.DataFrame:
    assert patch_level >= 0, "Patch level must be non-negative"
    assert slide.is_open, "Slide must be open to create patchset"

    # work out the size of the feature array
    size_in_patches = slide.size_in_patches(patch_size, patch_level)

    # if no detector is provided, we assume the entire slide is foreground
    if detector is None:
        tissue_mask = np.ones(size_in_patches.as_shape(), dtype=bool)
    else:
        tissue_mask = detector(slide)

    # convert the tissue mask into dataframes with location information
    tissue_df = to_frame_with_locations(tissue_mask, 'tissue_score')

    # if there are annotations, render them as a grid and convert to a dataframe with location information
    if annotation_set is not None and len(annotation_set) > 0:
        annotation_labels = annotation_set.render_to_grid(size_in_patches.as_shape(), patch_size, patch_level)
        annotation_df = to_frame_with_locations(annotation_labels, 'annotation_label')
    
        # merge the tissue and annotation dataframes on the patch locations
        patch_df = tissue_df.merge(annotation_df, on=['row', 'column'], how='left')
    else:
        patch_df = tissue_df
        patch_df['annotation_label'] = annotation_set.get_fill_index()

    return patch_df

def make_patchset(sample: Sample, patch_size: int, patch_level: int, detector: Detector | None = None) -> PatchSet:
    annotation_set = sample.make_annotations()  # will return None if there are no annotations, which is fine for make_patchset_df
    
    with sample.make_slide() as slide:
        patch_df = make_patchset_df(slide, patch_size, patch_level, detector, annotation_set)
    
    patch_df['context_id'] = 0  # for now we only have one context per patch, but this allows for multiple contexts in the future if needed
    patch_df['keep'] = patch_df['tissue_score'] > 0.03  # add a keep column based on the tissue score, can change later
    
    context = PatchContext(sample=sample, level=patch_level, patch_size=patch_size)
    
    manifest = PatchSetManifest(created_at=str(datetime.datetime.now().isoformat()))
    
    return PatchSet(frame=patch_df, contexts=[context], manifest=manifest)

pathsets: list[PatchSet] = []
detector = per_patch_canny_ranker(224, 1)
for sample in cerv_mini_dataset.samples():
    patchset = make_patchset(sample, patch_size=224, patch_level=1, detector=detector)
    pathsets.append(patchset)
    print(patchset.frame.head())
    print(patchset.describe())

In [ ]:
from histokit.viz.patchloc import visualise_patchset

for patchset in pathsets:
    visualise_patchset(patchset, width=1024)
